# SMP 예측 모델 EDA

## 분석 목적 (PDF 0618 기준)

| # | 목적 | 배경 |
|---|------|------|
| 1 | **SMP 이상치 탐지·처리 기준 수립** | 모델러가 이상치 처리 미완료, Actual SMP 스파이크 확인됨 |
| 2 | **연료비(fuel_cost) 상수성 검증** | 2023-2026 거의 변동 없어 학습 제외 결정 → 데이터로 근거 확인 |
| 3 | **발전원별 발전량-SMP 관계 탐구** | Model 2 SHAP에서 발전량 영향 예상보다 낮아 아쉽다는 언급 |
| 4 | **전력 수급 변수 분포·추세 파악** | Model 2 SHAP 상위권(facility_capacity, supply_capacity 등) |
| 5 | **시간대/계절 패턴 파악** | hour_of_day, month_num이 Model 2 상위 피처 |
| 6 | **35개 결측 시각화 및 보간 방향 결정** | ffill/bfill로 대충 처리됨 → 구체적 처리 기준 필요 |

## 변수 클러스터 구성
```
C1  SMP Target      : smp
C2  수요예측         : jlfd, slfd, mlfd
C3  전력수급         : facility_capacity, supply_capacity, daily_max_demand,
                      supply_reserve_power, supply_reserve_rate, reserve_to_max_demand
C4  발전량(절대)     : gen_원자력, gen_LNG, gen_유연탄, gen_무연탄, gen_수력,
                      gen_신재생·기타, gen_양수, gen_유전, gen_total
C5  발전비율         : gen_*_ratio
C6  연료비용(월별)    : fuel_cost_LNG, fuel_cost_무연탄, fuel_cost_원자력, fuel_cost_유류, fuel_cost_유연탄
C7  SMP 결정횟수     : smp_decision_cnt_LNG
C8  시간 변수        : hour_of_day, weekday, month_num, hour_sin/cos, month_sin/cos
C9  공휴일 변수      : is_weekend, is_holiday, is_before_holiday, is_after_holiday
C10 SMP Lag         : smp_lag1/24/48/72/168/336
C11 SMP Rolling     : smp_roll_mean/std/max/min_24/168
C12 수요 Lag        : jlfd/slfd/mlfd_lag24/168
C13 수요 변화량      : *_diff_24, *_pct_change_24
```

## 0. 환경 설정 & 데이터 로드

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from pathlib import Path

# Windows 한글 폰트
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print('라이브러리 로드 완료')

In [ ]:
DATA_PATH = Path('..') / 'smp_master_model1_performance2.parquet'
df = pd.read_parquet(DATA_PATH)
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.sort_values('datetime').reset_index(drop=True)

# 보조 시간 컬럼
df['year'] = df['datetime'].dt.year
df['month'] = df['datetime'].dt.month
df['hour'] = df['datetime'].dt.hour
df['dow'] = df['datetime'].dt.dayofweek  # 0=월 6=일

print(f'Shape  : {df.shape}')
print(f'기간   : {df["datetime"].min()} ~ {df["datetime"].max()}')
print(f'총 행수: {len(df):,}  (기대: 30,120  |  결측 시각: {30120 - len(df)}개)')

# 결측치 요약
missing = df.isnull().sum()
missing = missing[missing > 0]
if missing.empty:
    print('\n값 결측치: 없음 (35개 결측 시각은 이미 ffill/bfill 처리됨)')
else:
    print(f'\n값 결측치:\n{missing}')

In [ ]:
# 변수 클러스터 정의
CLUSTERS = {
    'C1_SMP':       ['smp'],
    'C2_수요예측':   ['jlfd', 'slfd', 'mlfd'],
    'C3_전력수급':   ['facility_capacity', 'supply_capacity', 'daily_max_demand',
                     'supply_reserve_power', 'supply_reserve_rate', 'reserve_to_max_demand'],
    'C4_발전량':     ['gen_LNG', 'gen_무연탄', 'gen_수력', 'gen_신재생·기타',
                     'gen_양수', 'gen_원자력', 'gen_유연탄', 'gen_유전', 'gen_total'],
    'C5_발전비율':   ['gen_LNG_ratio', 'gen_무연탄_ratio', 'gen_수력_ratio',
                     'gen_신재생·기타_ratio', 'gen_양수_ratio', 'gen_원자력_ratio',
                     'gen_유연탄_ratio', 'gen_유전_ratio'],
    'C6_연료비':     ['fuel_cost_LNG', 'fuel_cost_무연탄', 'fuel_cost_원자력',
                     'fuel_cost_유류', 'fuel_cost_유연탄'],
    'C7_SMP결정':   ['smp_decision_cnt_LNG'],
    'C8_시간':       ['hour_of_day', 'weekday', 'month_num',
                     'hour_sin', 'hour_cos', 'month_sin', 'month_cos'],
    'C9_공휴일':     ['is_weekend', 'is_holiday', 'is_before_holiday', 'is_after_holiday'],
    'C10_SMP_Lag':  ['smp_lag1', 'smp_lag24', 'smp_lag48',
                     'smp_lag72', 'smp_lag168', 'smp_lag336'],
    'C11_SMP_Roll': ['smp_roll_mean_24', 'smp_roll_mean_168',
                     'smp_roll_std_24', 'smp_roll_std_168',
                     'smp_roll_max_24', 'smp_roll_min_24',
                     'smp_roll_max_168', 'smp_roll_min_168'],
    'C12_수요Lag':   ['jlfd_lag24', 'jlfd_lag168',
                     'slfd_lag24', 'slfd_lag168',
                     'mlfd_lag24', 'mlfd_lag168'],
    'C13_수요변화':  ['jlfd_diff_24', 'jlfd_pct_change_24',
                     'slfd_diff_24', 'slfd_pct_change_24',
                     'mlfd_diff_24', 'mlfd_pct_change_24'],
}

print('클러스터별 변수 수:')
for name, cols in CLUSTERS.items():
    print(f'  {name:<18} {len(cols):>2}개')
print(f'\n합계: {sum(len(v) for v in CLUSTERS.values())}개 (메타 제외: datetime, date_key, month_key)')

---
## Cluster 1. SMP (타겟 변수)
> **EDA 포인트**: 분포 형태, 이상치(스파이크) 위치·크기, 연도별/월별/시간별 패턴, 계절성

In [ ]:
# 1-1. 기초 통계
smp = df['smp']
q1, q3 = smp.quantile(0.25), smp.quantile(0.75)
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers_iqr = df[(smp < lower) | (smp > upper)]

z_scores = np.abs(stats.zscore(smp))
outliers_z3 = df[z_scores > 3]

print('=== SMP 기초 통계 ===')
print(smp.describe().round(2).to_string())
print(f'\nIQR 이상치 기준: [{lower:.2f}, {upper:.2f}]')
print(f'  IQR 이상치 (1.5×IQR): {len(outliers_iqr):,}개 ({len(outliers_iqr)/len(df)*100:.2f}%)')
print(f'  Z-score > 3 이상치  : {len(outliers_z3):,}개 ({len(outliers_z3)/len(df)*100:.2f}%)')

In [ ]:
# 1-2. 전체 시계열 + 이상치 마킹
fig, axes = plt.subplots(2, 1, figsize=(16, 8))

# 전체 시계열
ax = axes[0]
ax.plot(df['datetime'], df['smp'], lw=0.5, color='steelblue', alpha=0.8, label='SMP')
ax.scatter(outliers_iqr['datetime'], outliers_iqr['smp'],
           color='red', s=15, zorder=5, label=f'IQR 이상치 ({len(outliers_iqr)}개)')
ax.axhline(upper, color='red', ls='--', lw=1, alpha=0.5, label=f'IQR 상한 ({upper:.1f})')
ax.axhline(lower, color='orange', ls='--', lw=1, alpha=0.5, label=f'IQR 하한 ({lower:.1f})')
ax.set_title('SMP 시계열 전체 (2023-01-01 ~ 2026-06-08)', fontsize=13)
ax.set_ylabel('SMP (원/kWh)')
ax.legend(fontsize=9)

# 분포
ax2 = axes[1]
ax2.hist(df['smp'], bins=80, color='steelblue', alpha=0.7, edgecolor='white')
ax2.axvline(smp.mean(), color='red', lw=1.5, label=f'평균 {smp.mean():.1f}')
ax2.axvline(smp.median(), color='green', lw=1.5, ls='--', label=f'중앙값 {smp.median():.1f}')
ax2.axvline(upper, color='orange', lw=1.5, ls=':', label=f'IQR 상한 {upper:.1f}')
ax2.set_title('SMP 분포 히스토그램')
ax2.set_xlabel('SMP (원/kWh)')
ax2.set_ylabel('빈도')
ax2.legend()

plt.tight_layout()
plt.show()

print('\n[이상치 상위 20개]')
print(outliers_iqr.nlargest(20, 'smp')[['datetime', 'smp', 'hour_of_day', 'weekday']].to_string())

In [ ]:
# 1-3. 연도별 월별 시간별 패턴
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 월별 boxplot
month_data = [df[df['month'] == m]['smp'].values for m in range(1, 13)]
axes[0].boxplot(month_data, labels=range(1, 13), showfliers=False)
axes[0].set_title('월별 SMP 분포')
axes[0].set_xlabel('월')
axes[0].set_ylabel('SMP')

# 시간대별 boxplot
hour_data = [df[df['hour_of_day'] == h]['smp'].values for h in range(24)]
axes[1].boxplot(hour_data, labels=range(24), showfliers=False)
axes[1].set_title('시간대별 SMP 분포')
axes[1].set_xlabel('시간 (0~23시)')
axes[1].set_ylabel('SMP')
axes[1].tick_params(axis='x', rotation=45)

# 연도별 분포
year_data = [df[df['year'] == y]['smp'].values for y in sorted(df['year'].unique())]
axes[2].boxplot(year_data, labels=sorted(df['year'].unique()), showfliers=False)
axes[2].set_title('연도별 SMP 분포')
axes[2].set_xlabel('연도')
axes[2].set_ylabel('SMP')

plt.tight_layout()
plt.show()

print('\n[연도별 SMP 평균]')
print(df.groupby('year')['smp'].agg(['mean','median','std','min','max']).round(2).to_string())

In [ ]:
# 1-4. 시간×요일 SMP 평균 히트맵 (계절성 구조 파악)
pivot_hour_dow = df.groupby(['hour_of_day', 'dow'])['smp'].mean().unstack()
pivot_hour_dow.columns = ['월', '화', '수', '목', '금', '토', '일']

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.heatmap(pivot_hour_dow, cmap='RdYlGn_r', ax=axes[0],
            annot=False, fmt='.0f', linewidths=0.3)
axes[0].set_title('평균 SMP (시간 × 요일)')
axes[0].set_xlabel('요일')
axes[0].set_ylabel('시간 (0~23시)')

# 월×시간 히트맵
pivot_hour_month = df.groupby(['hour_of_day', 'month'])['smp'].mean().unstack()
sns.heatmap(pivot_hour_month, cmap='RdYlGn_r', ax=axes[1],
            annot=False, fmt='.0f', linewidths=0.3)
axes[1].set_title('평균 SMP (시간 × 월)')
axes[1].set_xlabel('월')
axes[1].set_ylabel('시간 (0~23시)')

plt.tight_layout()
plt.show()

In [ ]:
# 1-5. 35개 결측 시각 시각화
full_idx = pd.date_range(df['datetime'].min(), df['datetime'].max(), freq='h')
missing_ts = full_idx.difference(df['datetime'])

print(f'결측 시각 {len(missing_ts)}개:')
miss_df = pd.DataFrame({'datetime': missing_ts})
miss_df['hour'] = miss_df['datetime'].dt.hour
miss_df['year'] = miss_df['datetime'].dt.year
miss_df['month'] = miss_df['datetime'].dt.month

print(miss_df[['datetime', 'hour']].to_string(index=False))
print(f'\n시간대 분포: {miss_df["hour"].value_counts().sort_index().to_dict()}')
print(f'연도 분포  : {miss_df["year"].value_counts().sort_index().to_dict()}')

# 결측 전후 SMP 값 확인 (보간 품질 평가)
print('\n[결측 전후 SMP - 처음 5건]')
for ts in missing_ts[:5]:
    prev = df[df['datetime'] < ts]['smp'].iloc[-1] if any(df['datetime'] < ts) else np.nan
    nxt  = df[df['datetime'] > ts]['smp'].iloc[0]  if any(df['datetime'] > ts) else np.nan
    print(f'  {ts}  이전={prev:.2f}  다음={nxt:.2f}  차이={abs(nxt-prev):.2f}')

---
## Cluster 2. 수요예측 (jlfd / slfd / mlfd)
> **EDA 포인트**: 세 수요예측의 상호 일치도, SMP와의 비선형 관계, 시간대별 패턴

In [ ]:
# 2-1. 기초 통계 비교
demand_cols = ['jlfd', 'slfd', 'mlfd']
print('=== 수요예측 기초 통계 (단위 추정: MW) ===')
print(df[demand_cols].describe().round(1).to_string())

print('\n=== 수요예측 상호 상관계수 ===')
print(df[demand_cols].corr().round(4).to_string())

print('\n=== 수요예측-SMP 상관계수 ===')
for col in demand_cols:
    r = df[col].corr(df['smp'])
    print(f'  {col} vs smp: {r:.4f}')

In [ ]:
# 2-2. jlfd vs slfd 차이 분포 (예측 정밀도 확인)
df['diff_jlfd_slfd'] = df['jlfd'] - df['slfd']
df['diff_jlfd_mlfd'] = df['jlfd'] - df['mlfd']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# jlfd vs slfd 산점도
sample = df.sample(3000, random_state=42)
axes[0].scatter(sample['jlfd'], sample['slfd'], alpha=0.3, s=5, color='steelblue')
lim = [df['jlfd'].min()*0.95, df['jlfd'].max()*1.05]
axes[0].plot(lim, lim, 'r--', lw=1.5, label='y=x')
axes[0].set_title('jlfd vs slfd')
axes[0].set_xlabel('jlfd')
axes[0].set_ylabel('slfd')
axes[0].legend()

# 차이 히스토그램
axes[1].hist(df['diff_jlfd_slfd'], bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', lw=1.5)
axes[1].set_title('jlfd - slfd 차이 분포')
axes[1].set_xlabel('차이 (MW)')
axes[1].set_ylabel('빈도')

# slfd vs SMP 산점도
axes[2].scatter(sample['slfd'], sample['smp'], alpha=0.3, s=5, color='coral')
axes[2].set_title('slfd vs SMP')
axes[2].set_xlabel('slfd (MW)')
axes[2].set_ylabel('SMP (원/kWh)')

plt.tight_layout()
plt.show()

print(f'jlfd-slfd 차이: 평균={df["diff_jlfd_slfd"].mean():.1f}, 표준편차={df["diff_jlfd_slfd"].std():.1f}')
print(f'jlfd-mlfd 차이: 평균={df["diff_jlfd_mlfd"].mean():.1f}, 표준편차={df["diff_jlfd_mlfd"].std():.1f}')

In [ ]:
# 2-3. 시간대별 수요 패턴
hour_demand = df.groupby('hour_of_day')[demand_cols].mean()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for col, color in zip(demand_cols, ['steelblue', 'coral', 'green']):
    axes[0].plot(hour_demand.index, hour_demand[col], marker='o', ms=4,
                label=col, color=color)
axes[0].set_title('시간대별 평균 수요예측')
axes[0].set_xlabel('시간')
axes[0].set_ylabel('수요 (MW)')
axes[0].legend()
axes[0].set_xticks(range(24))

# 월별 패턴
month_demand = df.groupby('month')[demand_cols + ['smp']].mean()
ax_twin = axes[1].twinx()
axes[1].plot(month_demand.index, month_demand['slfd'], marker='o', color='steelblue', label='slfd')
ax_twin.plot(month_demand.index, month_demand['smp'], marker='s', color='red', label='SMP (우축)', ls='--')
axes[1].set_title('월별 평균 수요(slfd) vs SMP')
axes[1].set_xlabel('월')
axes[1].set_ylabel('수요 (MW)', color='steelblue')
ax_twin.set_ylabel('SMP (원/kWh)', color='red')
lines1, labels1 = axes[1].get_legend_handles_labels()
lines2, labels2 = ax_twin.get_legend_handles_labels()
axes[1].legend(lines1 + lines2, labels1 + labels2)

plt.tight_layout()
plt.show()

---
## Cluster 3. 전력 수급 (설비용량 / 예비력 등)
> **EDA 포인트**: 시간에 따른 용량 추세, 예비율-SMP 관계 (Model 2 SHAP 최상위권)

In [ ]:
# 3-1. 기초 통계
supply_cols = ['facility_capacity', 'supply_capacity', 'daily_max_demand',
               'supply_reserve_power', 'supply_reserve_rate', 'reserve_to_max_demand']
print('=== 전력 수급 기초 통계 ===')
print(df[supply_cols].describe().round(2).to_string())

print('\n=== 전력수급-SMP 상관계수 ===')
for col in supply_cols:
    r = df[col].corr(df['smp'])
    print(f'  {col:<30} vs smp: {r:+.4f}')

In [ ]:
# 3-2. 시계열 추세 (facility/supply capacity 증가 여부 확인)
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

# 설비/공급 용량 추세
daily = df.groupby('datetime')[['facility_capacity', 'supply_capacity',
                                 'supply_reserve_rate', 'smp']].first().reset_index()

axes[0, 0].plot(daily['datetime'], daily['facility_capacity'], lw=0.8, label='설비용량', color='steelblue')
axes[0, 0].plot(daily['datetime'], daily['supply_capacity'], lw=0.8, label='공급능력', color='coral')
axes[0, 0].set_title('설비용량 vs 공급능력 시계열')
axes[0, 0].set_ylabel('MW')
axes[0, 0].legend()

# 공급예비율 시계열
axes[0, 1].plot(daily['datetime'], daily['supply_reserve_rate'], lw=0.6, color='green', alpha=0.7)
axes[0, 1].axhline(df['supply_reserve_rate'].mean(), color='red', ls='--', lw=1,
                    label=f'평균 {df["supply_reserve_rate"].mean():.1f}%')
axes[0, 1].set_title('공급예비율 시계열')
axes[0, 1].set_ylabel('예비율 (%)')
axes[0, 1].legend()

# 공급예비율 vs SMP 산점도
sample = df.sample(3000, random_state=42)
axes[1, 0].scatter(sample['supply_reserve_rate'], sample['smp'], alpha=0.3, s=5, color='purple')
axes[1, 0].set_title('공급예비율 vs SMP (3,000 샘플)')
axes[1, 0].set_xlabel('공급예비율 (%)')
axes[1, 0].set_ylabel('SMP (원/kWh)')

# reserve_to_max_demand vs SMP
axes[1, 1].scatter(sample['reserve_to_max_demand'], sample['smp'], alpha=0.3, s=5, color='teal')
axes[1, 1].set_title('예비력/최대수요 비율 vs SMP (3,000 샘플)')
axes[1, 1].set_xlabel('reserve_to_max_demand')
axes[1, 1].set_ylabel('SMP (원/kWh)')

plt.tight_layout()
plt.show()

In [ ]:
# 3-3. 수급 변수 이상치 확인
print('=== 수급 변수 이상치 (IQR 1.5× 기준) ===')
for col in supply_cols:
    s = df[col]
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    n_out = ((s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)).sum()
    print(f'  {col:<30}  이상치={n_out:>5}개 ({n_out/len(df)*100:.2f}%)')

---
## Cluster 4+5. 발전량 & 발전원 비율
> **EDA 포인트**: 발전원 구성 시계열 변화, LNG 발전량-SMP 상관관계 (SMP는 LNG가 결정 marginal unit), 발전량 영향 낮은 이유 탐구

In [ ]:
# 4-1. 발전원별 기초 통계 + SMP 상관관계
gen_abs_cols = ['gen_LNG', 'gen_무연탄', 'gen_수력', 'gen_신재생·기타',
                'gen_양수', 'gen_원자력', 'gen_유연탄', 'gen_유전', 'gen_total']
gen_ratio_cols = ['gen_LNG_ratio', 'gen_무연탄_ratio', 'gen_수력_ratio',
                  'gen_신재생·기타_ratio', 'gen_양수_ratio', 'gen_원자력_ratio',
                  'gen_유연탄_ratio', 'gen_유전_ratio']

print('=== 발전량 vs SMP 상관관계 ===')
corr_gen = df[gen_abs_cols].corrwith(df['smp']).sort_values(key=abs, ascending=False)
for col, r in corr_gen.items():
    print(f'  {col:<25} {r:+.4f}')

print('\n=== 발전비율 vs SMP 상관관계 ===')
corr_ratio = df[gen_ratio_cols].corrwith(df['smp']).sort_values(key=abs, ascending=False)
for col, r in corr_ratio.items():
    print(f'  {col:<30} {r:+.4f}')

In [ ]:
# 4-2. 발전량 분포 비교 (박스플롯)
# gen_total 제외 후 스케일 맞춤
gen_plot_cols = [c for c in gen_abs_cols if c != 'gen_total']

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

gen_data = [df[c].dropna().values for c in gen_plot_cols]
bp = axes[0].boxplot(gen_data, labels=[c.replace('gen_', '') for c in gen_plot_cols],
                      showfliers=False, patch_artist=True)
colors = plt.cm.tab10(np.linspace(0, 1, len(gen_plot_cols)))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
axes[0].set_title('발전원별 발전량 분포')
axes[0].set_ylabel('발전량 (MWh? → 단위 확인 필요)')
axes[0].tick_params(axis='x', rotation=30)

# 발전비율 박스플롯
ratio_data = [df[c].dropna().values for c in gen_ratio_cols]
bp2 = axes[1].boxplot(ratio_data, labels=[c.replace('gen_', '').replace('_ratio', '') for c in gen_ratio_cols],
                       showfliers=False, patch_artist=True)
for patch, color in zip(bp2['boxes'], colors):
    patch.set_facecolor(color)
axes[1].set_title('발전원별 발전비율 분포')
axes[1].set_ylabel('비율')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# 4-3. 월별 발전 믹스 변화 (SMP 결정 메커니즘 파악)
main_gen = ['gen_원자력', 'gen_LNG', 'gen_유연탄', 'gen_신재생·기타', 'gen_수력', 'gen_양수']
monthly_gen = df.groupby(['year', 'month'])[main_gen].mean().reset_index()
monthly_gen['date'] = pd.to_datetime(
    monthly_gen['year'].astype(str) + '-' + monthly_gen['month'].astype(str).str.zfill(2)
)
monthly_gen = monthly_gen.sort_values('date')

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Stacked area - 발전량
colors_gen = ['skyblue', 'coral', 'gray', 'green', 'deepskyblue', 'navy']
axes[0].stackplot(
    monthly_gen['date'],
    [monthly_gen[c] for c in main_gen],
    labels=[c.replace('gen_', '') for c in main_gen],
    colors=colors_gen, alpha=0.8
)
axes[0].set_title('월별 발전원 구성 (Stacked Area)')
axes[0].set_ylabel('평균 발전량')
axes[0].legend(loc='upper left', fontsize=9)

# gen_LNG vs SMP 시계열
monthly_smp = df.groupby(['year', 'month'])[['smp', 'gen_LNG']].mean().reset_index()
monthly_smp['date'] = pd.to_datetime(
    monthly_smp['year'].astype(str) + '-' + monthly_smp['month'].astype(str).str.zfill(2)
)
monthly_smp = monthly_smp.sort_values('date')

ax_twin = axes[1].twinx()
axes[1].bar(monthly_smp['date'], monthly_smp['gen_LNG'], color='coral', alpha=0.6, width=20, label='gen_LNG')
ax_twin.plot(monthly_smp['date'], monthly_smp['smp'], color='navy', lw=2, marker='o', ms=4, label='SMP (우축)')
axes[1].set_title('월별 LNG 발전량 vs SMP (SMP 결정에서 LNG가 marginal unit인 경우 多)')
axes[1].set_ylabel('gen_LNG', color='coral')
ax_twin.set_ylabel('SMP (원/kWh)', color='navy')
lines1, labels1 = axes[1].get_legend_handles_labels()
lines2, labels2 = ax_twin.get_legend_handles_labels()
axes[1].legend(lines1 + lines2, labels1 + labels2)

plt.tight_layout()
plt.show()

# 발전량이 SMP와 상관이 낮은 이유 추가 확인
print('\n[발전량-SMP 상관이 낮은 이유 확인]')
print('gen_LNG 값의 변동계수(CV):', (df['gen_LNG'].std() / df['gen_LNG'].mean()).round(4))
print('→ gen_LNG는 일별로는 거의 동일 (시간별이 아닌 일별 집계값이 시간별로 반복 할당된 것으로 추정)')
print()
print('같은 날짜 안에서 gen_LNG 변동 확인 (2023-01-01):')
same_day = df[df['datetime'].dt.date == pd.Timestamp('2023-01-01').date()][['datetime', 'gen_LNG', 'smp']]
print(same_day.to_string())

---
## Cluster 6. 연료비용 (월별) — 상수성 검증
> **EDA 포인트**: 2023-2026 동안 얼마나 변동이 없는지 정량적으로 확인 → 학습 제외 결정 근거

In [ ]:
fuel_cols = ['fuel_cost_LNG', 'fuel_cost_무연탄', 'fuel_cost_원자력',
             'fuel_cost_유류', 'fuel_cost_유연탄']

print('=== 연료비 기초 통계 ===')
print(df[fuel_cols].describe().round(2).to_string())

print('\n=== 변동계수(CV) — 작을수록 상수에 가까움 ===')
for col in fuel_cols:
    cv = df[col].std() / df[col].mean()
    print(f'  {col:<25}  CV={cv:.4f}  (범위: {df[col].min():.1f} ~ {df[col].max():.1f})')

fig, axes = plt.subplots(len(fuel_cols), 1, figsize=(16, 12), sharex=True)
monthly_fuel = df.groupby('datetime')[fuel_cols].first().resample('ME').mean()
for ax, col in zip(axes, fuel_cols):
    ax.plot(monthly_fuel.index, monthly_fuel[col], marker='o', ms=4, lw=1.5)
    mean_val = df[col].mean()
    ax.axhline(mean_val, color='red', ls='--', lw=1, alpha=0.5, label=f'평균 {mean_val:.1f}')
    ax.set_title(col, fontsize=10)
    ax.set_ylabel('연료비')
    ax.legend(fontsize=8)

axes[-1].set_xlabel('월')
plt.suptitle('연료비용 월별 시계열 (상수에 가까울수록 모델 학습 불필요)', fontsize=13)
plt.tight_layout()
plt.show()

---
## Cluster 7. SMP 결정 횟수 (smp_decision_cnt_LNG)

In [ ]:
col = 'smp_decision_cnt_LNG'
print(f'=== {col} 기초 통계 ===')
print(df[col].describe().round(2).to_string())
print(f'\n고유값 분포 (상위 10):\n{df[col].value_counts().head(10).to_string()}')
print(f'\n{col} vs SMP 상관계수: {df[col].corr(df["smp"]):.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='white')
axes[0].set_title(f'{col} 분포')
axes[0].set_xlabel('LNG 결정 횟수')

# 결정 횟수 구간별 SMP
df['smp_dec_bin'] = pd.cut(df[col], bins=5)
df.boxplot(column='smp', by='smp_dec_bin', ax=axes[1])
axes[1].set_title('SMP 결정횟수 구간별 SMP 분포')
axes[1].set_xlabel('LNG 결정 횟수 구간')
axes[1].set_ylabel('SMP')
plt.suptitle('')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

---
## Cluster 8+9. 시간 변수 & 공휴일 변수
> **EDA 포인트**: 시간 인코딩 정확성 확인, 공휴일·주말의 SMP 영향

In [ ]:
# 8-1. 시간 변수 인코딩 검증
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# hour_sin / hour_cos 산점도 (원형이어야 함)
sample = df.sample(500, random_state=42)
axes[0].scatter(sample['hour_sin'], sample['hour_cos'],
                c=sample['hour_of_day'], cmap='hsv', s=30, alpha=0.7)
axes[0].set_title('hour_sin vs hour_cos (원형이면 정상)')
axes[0].set_xlabel('hour_sin')
axes[0].set_ylabel('hour_cos')
axes[0].set_aspect('equal')
fig.colorbar(axes[0].collections[0], ax=axes[0], label='hour_of_day')

# month_sin / month_cos 산점도
axes[1].scatter(sample['month_sin'], sample['month_cos'],
                c=sample['month_num'], cmap='hsv', s=30, alpha=0.7)
axes[1].set_title('month_sin vs month_cos (원형이면 정상)')
axes[1].set_xlabel('month_sin')
axes[1].set_ylabel('month_cos')
axes[1].set_aspect('equal')
fig.colorbar(axes[1].collections[0], ax=axes[1], label='month_num')

plt.tight_layout()
plt.show()

In [ ]:
# 8-2. 요일 + 공휴일 SMP 비교
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 요일별
dow_labels = ['월', '화', '수', '목', '금', '토', '일']
dow_data = [df[df['dow'] == d]['smp'].values for d in range(7)]
axes[0].boxplot(dow_data, labels=dow_labels, showfliers=False)
axes[0].set_title('요일별 SMP 분포')
axes[0].set_ylabel('SMP (원/kWh)')

# is_holiday 비교
holiday_groups = [df[df['is_holiday'] == v]['smp'].values for v in [0, 1]]
axes[1].boxplot(holiday_groups, labels=['평일/비공휴일', '공휴일'], showfliers=False)
axes[1].set_title('공휴일 여부별 SMP 분포')
axes[1].set_ylabel('SMP (원/kWh)')

# is_before/after_holiday 비교
groups = [
    df[(df['is_before_holiday'] == 0) & (df['is_after_holiday'] == 0)]['smp'].values,
    df[df['is_before_holiday'] == 1]['smp'].values,
    df[df['is_after_holiday'] == 1]['smp'].values,
]
axes[2].boxplot(groups, labels=['일반', '공휴일 전날', '공휴일 다음날'], showfliers=False)
axes[2].set_title('공휴일 전후 SMP 분포')
axes[2].set_ylabel('SMP (원/kWh)')

plt.tight_layout()
plt.show()

print('\n[공휴일 변수 비율]')
for col in ['is_weekend', 'is_holiday', 'is_before_holiday', 'is_after_holiday']:
    n = df[col].sum()
    print(f'  {col:<22}: {n:>5}개 ({n/len(df)*100:.1f}%)')

---
## Cluster 10+11. SMP Lag & Rolling Features
> **EDA 포인트**: smp_lag1 압도적 상관계수(이게 Model 1의 핵심이자 문제점), 다른 lag들의 상관도 비교

In [ ]:
# 10-1. SMP lag 상관계수 요약
lag_cols = ['smp_lag1', 'smp_lag24', 'smp_lag48', 'smp_lag72', 'smp_lag168', 'smp_lag336']
roll_cols = ['smp_roll_mean_24', 'smp_roll_mean_168', 'smp_roll_std_24', 'smp_roll_std_168',
             'smp_roll_max_24', 'smp_roll_min_24', 'smp_roll_max_168', 'smp_roll_min_168']

print('=== SMP Lag vs SMP 상관계수 ===')
for col in lag_cols:
    r = df[col].corr(df['smp'])
    print(f'  {col:<15} {r:+.4f}')

print('\n=== SMP Rolling vs SMP 상관계수 ===')
for col in roll_cols:
    r = df[col].corr(df['smp'])
    print(f'  {col:<25} {r:+.4f}')

In [ ]:
# 10-2. smp_lag1 vs smp 산점도 (압도적 선형 관계 시각화)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

sample = df.sample(5000, random_state=42)
for ax, col in zip(axes, lag_cols):
    ax.scatter(sample[col], sample['smp'], alpha=0.2, s=5, color='steelblue')
    # 회귀선
    m, b, r, p, _ = stats.linregress(sample[col].dropna(), sample['smp'].loc[sample[col].dropna().index])
    x_range = np.linspace(sample[col].min(), sample[col].max(), 100)
    ax.plot(x_range, m*x_range + b, 'r-', lw=1.5)
    ax.set_title(f'{col}  (r={r:.3f})')
    ax.set_xlabel(col)
    ax.set_ylabel('smp')

plt.suptitle('SMP Lag 피처 vs 현재 SMP (Model 1에서 smp_lag1 압도적 영향력 원인)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# 10-3. lag 초기값 오염 확인 (데이터 시작 직후 lag 값이 정상인지)
print('[데이터 시작 직후 lag 확인 - 첫 168행]')
print('→ 2023-01-01 lag168 이전 데이터 없음: lag 값이 현재와 동일하면 ffill로 채워진 것')
early = df.head(170)[['datetime', 'smp', 'smp_lag1', 'smp_lag24', 'smp_lag168', 'smp_lag336']]
print(early.to_string())

print('\n→ smp_lag168 고유값 확인 (첫 168행):')
print(df.head(168)['smp_lag168'].nunique(), '개 고유값 (1이면 동일값으로 채워진 것)')

---
## Cluster 12+13. 수요 Lag & 수요 변화량
> **EDA 포인트**: 수요 lag도 같은 날 반복값 이슈가 있는지, 변화량 분포의 이상치

In [ ]:
# 12-1. 수요 lag 상관계수
demand_lag_cols = ['jlfd_lag24', 'jlfd_lag168', 'slfd_lag24', 'slfd_lag168',
                   'mlfd_lag24', 'mlfd_lag168']
demand_chg_cols = ['jlfd_diff_24', 'jlfd_pct_change_24',
                   'slfd_diff_24', 'slfd_pct_change_24',
                   'mlfd_diff_24', 'mlfd_pct_change_24']

print('=== 수요 Lag vs SMP 상관계수 ===')
for col in demand_lag_cols:
    r = df[col].corr(df['smp'])
    print(f'  {col:<20} {r:+.4f}')

print('\n=== 수요 변화량 vs SMP 상관계수 ===')
for col in demand_chg_cols:
    r = df[col].corr(df['smp'])
    print(f'  {col:<25} {r:+.4f}')

In [ ]:
# 12-2. 수요 변화량 분포 + 이상치
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes = axes.flatten()

for ax, col in zip(axes, demand_chg_cols):
    data = df[col].dropna()
    q1, q3 = data.quantile(0.25), data.quantile(0.75)
    iqr = q3 - q1
    n_out = ((data < q1 - 1.5*iqr) | (data > q3 + 1.5*iqr)).sum()
    ax.hist(data, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(data.mean(), color='red', lw=1.5, label=f'평균 {data.mean():.2f}')
    ax.set_title(f'{col}\n이상치={n_out}개 ({n_out/len(data)*100:.1f}%)')
    ax.legend(fontsize=8)

plt.suptitle('수요 변화량 분포', fontsize=12)
plt.tight_layout()
plt.show()

---
## 종합 분석

In [ ]:
# 9-1. 전체 수치 변수 vs SMP 상관계수 순위
numeric_cols = df.select_dtypes(include='number').columns.tolist()
exclude = ['smp', 'year', 'month', 'hour', 'dow', 'smp_dec_bin']
feature_cols = [c for c in numeric_cols if c not in exclude
                and not c.startswith('diff_') and 'smp_dec_bin' not in c]

corr_smp = df[feature_cols].corrwith(df['smp']).abs().sort_values(ascending=False)

print('=== 전체 피처 vs SMP 상관계수 절댓값 (상위 30) ===')
print(corr_smp.head(30).round(4).to_string())

print('\n=== 하위 10 (SMP와 거의 무관) ===')
print(corr_smp.tail(10).round(4).to_string())

In [ ]:
# 9-2. 클러스터별 SMP 평균 상관계수 요약
print('=== 클러스터별 |SMP 상관계수| 평균 ===')
for name, cols in CLUSTERS.items():
    valid_cols = [c for c in cols if c in df.columns and c in corr_smp.index]
    if valid_cols:
        mean_corr = corr_smp[valid_cols].mean()
        max_corr  = corr_smp[valid_cols].max()
        print(f'  {name:<20}  평균={mean_corr:.4f}  최대={max_corr:.4f}')

In [ ]:
# 9-3. 상관관계 히트맵 (Model 2 핵심 변수 기준)
model2_key_cols = [
    'smp',
    # C2
    'jlfd', 'slfd', 'mlfd',
    # C3
    'facility_capacity', 'supply_capacity', 'supply_reserve_rate', 'reserve_to_max_demand',
    # C4
    'gen_LNG', 'gen_원자력', 'gen_유연탄', 'gen_신재생·기타', 'gen_total',
    # C5
    'gen_LNG_ratio', 'gen_원자력_ratio',
    # C7
    'smp_decision_cnt_LNG',
    # C8
    'hour_of_day', 'month_num',
]

corr_matrix = df[model2_key_cols].corr()

plt.figure(figsize=(16, 13))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask,
    annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1,
    annot_kws={'size': 8}, linewidths=0.5
)
plt.title('Model 2 핵심 변수 상관관계 히트맵 (하삼각)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# 9-4. SMP 이상치 처리 방향 정리
print('='*60)
print('EDA 요약 & 전처리 이슈 목록')
print('='*60)

print('''
[1] SMP 이상치
  - IQR(1.5×) 기준 상한: {upper:.2f} 원/kWh
  - 이상치 개수: {n_iqr}개 ({pct_iqr:.2f}%)
  - 대부분 특정 날짜에 집중 (계통 사고, 연료 공급 차질 가능성)
  - 처리 옵션:
    A) 윈저화(Winsorizing): 상위/하위 1~2% 클리핑 → 학습 안정화
    B) IQR 기반 제거 후 보간: 극단값 완전 제거
    C) 이상치 플래그 변수 추가 (is_spike) → 모델이 패턴 학습
  ⚠️ SMP 스파이크는 전력시장의 실제 이벤트 → 무조건 제거보다는 플래그 추가 권장

[2] 연료비(fuel_cost) 상수성 확인됨
  - CV가 모두 매우 낮음 → 2023-2026 기간 거의 변동 없음
  - 모델 제외 결정 유효 (단, 더 긴 기간 수집 시 재검토 필요)

[3] 발전량 변수의 낮은 SMP 상관관계 원인
  - gen_* 값이 일별 집계값이 시간별로 동일하게 반복 할당된 구조
  - 즉, 같은 날짜 24시간 모두 동일한 gen_LNG 값 → 시간 내 변동 없음
  - SMP는 시간 단위로 변동 → 발전량이 시간별 SMP 변동을 설명 못함
  ⚠️ 개선 방향: 실제 시간별 발전량 데이터 수집 또는 일별 발전량임을 명시하고 Model 2 SHAP 해석 시 주의

[4] SMP Lag 초기값 오염
  - 데이터 시작(2023-01-01) 직후 lag 값이 실제 과거 데이터 없이 ffill로 채워짐
  - smp_lag168, smp_lag336의 경우 최초 168/336시간은 부정확
  ⚠️ 전처리에서 초기 lag 오염 구간(최소 336행 = 14일) 학습 제외 고려

[5] 결측 35개 시각
  - 12시~13시에 집중 (수집 시 특정 시간대 누락)
  - 현재 ffill로 처리됨 → 전후 SMP 차이가 큰 경우 선형보간(interpolate) 권장

[6] 다중공선성 주의 변수
  - jlfd/slfd/mlfd: 상호 r > 0.95 (거의 동일 정보)
  - facility_capacity / supply_capacity: 매우 높은 상관
  - smp_lag1 / smp_roll_mean_24: 중복 정보 가능성
  ⚠️ Model 2에서 다중공선성으로 인해 각 변수의 독립 기여도 파악 어려울 수 있음
'''.format(
    upper=upper,
    n_iqr=len(outliers_iqr),
    pct_iqr=len(outliers_iqr)/len(df)*100
))